# 04 — Learn the response and profiling as likelihood optimization

We now have a simulator, a learned unbinned density model, and a deliberately imperfect signal model. This notebook solves two different optimization problems with neural networks:

1. **Population response:** physical signal strength $\mu\mapsto(g(\mu),a(\mu))$, the parameters that best fit the simulator population with the imperfect model.
2. **Experiment-level profiling:** the full observed experiment and a tested $\nu\mapsto\widehat\alpha_\nu$, together with a global estimator $(\widehat\nu,\widehat\alpha)$.

Both networks learn by **minimizing the negative log likelihood at their own outputs**. We do not generate numerical-fit labels for training. Direct fits are reserved for validation. The second task uses ordinary full unbinned experiments; there is no $N=2$ training stage and no quadratic assumption about the likelihood or statistic.

The precedent [Learning Optimal Test Statistics in the Presence of Nuisance Parameters](https://arxiv.org/abs/2203.13079) learns statistics through a classification objective with best-average-power motivation. Our objective here is different: it directly amortizes the specified likelihood optimizations.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

RUN_NAME = "default"  # Use a new name when changing the model or training setup.
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("Run:", RUN)
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import load_config, bad_anchors, fit, nll
from sampling import sample_process
from utils import HybridModel
from inference import (ResponseNetwork, ProfileNetwork, Inference, train_response,
                       train_profiles, make_training_experiments, response_validation,
                       validate_toys, save_inference)

config = load_config(RUN)
misspecification = json.loads((RUN / "misspecification.json").read_text())
epsilon = float(misspecification["epsilon"])
device = "cuda" if torch.cuda.is_available() else "cpu"
hybrid = HybridModel.load(RUN, device=device)
OUT = RUN / "inference"
OUT.mkdir(parents=True, exist_ok=True)
spec = {"config": config, "epsilon": epsilon, "hybrid_model_id": hybrid.model_id}
spec_path = OUT / "run_spec.json"
if spec_path.exists() and json.loads(spec_path.read_text()) != spec:
    raise ValueError("The model changed. Use a new RUN directory for this inference study.")
spec_path.write_text(json.dumps(spec, indent=2))
print("Device:", device, "| epsilon:", epsilon, "| density model:", hybrid.model_id)

## 1. The population response network

Let $R_c(x;\alpha)$ denote a complete process/reference ratio in the **bad likelihood**, and write its intensity ratio as

$$I(x;\nu,\alpha)=\nu S R_S(x;\alpha)+B R_B(x;\alpha).$$

Our target is

$$(g(\mu),a(\mu))=\arg\min_{\nu,\alpha}\left[\nu S+B-\mu S\,\mathbb E_{p_{S,\mathrm{sim}}}\log I(X;\nu,\alpha)-B\,\mathbb E_{p_{B,\mathrm{sim}}}\log I(X;\nu,\alpha)+\frac{\alpha^2}{2}\right].$$

The generating nuisance is fixed at $\alpha_{\mathrm{gen}}=0$. A Gaussian constraint enters the likelihood, while the specified toy law is $Y\sim\mathrm{Uniform}[-2,2]$. Because $\mathbb E[Y]=0$, averaging $(Y-\alpha)^2/2$ leaves $\alpha^2/2$ plus a parameter-independent constant. This is the response for that explicitly chosen ensemble, not the usual Gaussian-auxiliary ensemble.

The response NN receives **only $\mu$** and outputs $(\nu,\alpha)$. The density ratios enter its loss. A vector of physical parameters would simply become a vector input; varying generating nuisances would require including them in the input too.

Notebook 03 saved separate physical signal and background samples, with the bad model's ratios evaluated on them. Their weighted averages approximate the two simulator expectations above. These are not samples from the learned good density model. We use the whole finite bank at each step, sample only $\mu$, and subtract a parameter-independent baseline from the loss for easier plotting. Bounds are $0\leq\nu\leq3$ and $-1\leq\alpha\leq1$; the sigmoid/tanh outputs approach their endpoints, so endpoint accuracy is checked below.

In [ ]:
bank_metadata = json.loads((RUN / "misspecified/response_bank.json").read_text())
assert bank_metadata["epsilon"] == epsilon
assert bank_metadata["hybrid_model_id"] == hybrid.model_id
signal_bank = np.load(RUN / "misspecified/response_signal.npy")
background_bank = np.load(RUN / "misspecified/response_background.npy")
print("Population quadrature events:", len(signal_bank), "signal +", len(background_bank), "background")

RETRAIN_RESPONSE = False
RESPONSE_STEPS = 2000
response_path = OUT / f"response_{RESPONSE_STEPS}steps.pt"
response_history_path = response_path.with_suffix(".json")
if RETRAIN_RESPONSE or not response_path.exists():
    response_network, response_history = train_response(
        signal_bank, background_bank, config, steps=RESPONSE_STEPS,
        batch_mu=8, device=device, seed=config["seed"] + 4101)
    torch.save(response_network.cpu().state_dict(), response_path)
    response_history_path.write_text(json.dumps(response_history, indent=2))
else:
    response_network = ResponseNetwork(config)
    response_network.load_state_dict(torch.load(response_path, map_location="cpu", weights_only=True))
    response_history = json.loads(response_history_path.read_text())
response_network = response_network.to(device)

history = pd.DataFrame(response_history)
plt.plot(history.step, history.monitor)
plt.xlabel("Training step")
plt.ylabel("Population NLL minus fixed baseline")
plt.title("Fixed mu-grid monitor on the training quadrature bank")
plt.show()

### Separate optimization error from finite-population Monte Carlo error

A direct fit on the **same** bank tests whether the NN minimized its training objective. A direct fit on an **independent physical bank** tests whether the response transfers to a second approximation of the simulator expectation. Differences between those two direct fits are finite-bank Monte Carlo variation, which is especially relevant at $S/B=1/100$.

We generate the validation bank once and reuse it only for diagnostics. Neither those events nor the resulting direct fits train the response NN. If same-bank likelihood gaps are small but the two direct-fit curves disagree, increase the population banks before increasing NN capacity.

In [ ]:
N_RESPONSE_VALIDATION = 200_000  # per physical process; independent of notebook 03
validation_bank_path = OUT / "response_validation_bank.npz"
if not validation_bank_path.exists():
    rng = np.random.default_rng(config["seed"] + 4201)
    validation_signal = bad_anchors(hybrid.anchors(sample_process("signal", N_RESPONSE_VALIDATION, rng)), epsilon)
    validation_background = bad_anchors(hybrid.anchors(sample_process("background", N_RESPONSE_VALIDATION, rng)), epsilon)
    np.savez(validation_bank_path, signal=validation_signal, background=validation_background)
else:
    bank = np.load(validation_bank_path)
    validation_signal, validation_background = bank["signal"], bank["background"]

mu_grid = np.linspace(*config["mu_range"], 13)
response_train_check = pd.DataFrame(response_validation(response_network, signal_bank, background_bank, mu_grid, config))
response_check = pd.DataFrame(response_validation(response_network, validation_signal, validation_background, mu_grid, config))
response_train_check.to_csv(OUT / "response_same_bank.csv", index=False)
response_check.to_csv(OUT / "response_independent_bank.csv", index=False)
display(response_check)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, label in [(axes[0], "nu", r"$g(\mu)$"), (axes[1], "alpha", r"$a(\mu)$")]:
    ax.plot(mu_grid, response_check[name + "_nn"], label="Response NN")
    ax.plot(mu_grid, response_train_check[name + "_exact"], "o", label="Direct fit: training bank")
    ax.plot(mu_grid, response_check[name + "_exact"], "x", label="Direct fit: independent bank")
    ax.set(xlabel=r"$\mu$", ylabel=label)
axes[0].legend(fontsize=8)
axes[2].plot(mu_grid, response_train_check.nll_gap, "o-", label="Same bank")
axes[2].plot(mu_grid, response_check.nll_gap, "x-", label="Independent bank")
axes[2].set(xlabel=r"$\mu$", ylabel="NN NLL − direct minimum")
axes[2].legend()
plt.tight_layout()
plt.show()

## 2. Represent the full experiment and learn both optimizers

Each event contributes the six normalized anchor ratios

$$\mathbf R(x)=(R_S^-,R_S^0,R_S^+,R_B^-,R_B^0,R_B^+).$$

Together with the known rates, count $N$, and auxiliary $y$, these determine the likelihood under our specified linear interpolation. The event encoder $\phi$ receives $\log(1+\mathbf R(x))$ and learns a permutation-invariant experiment representation:

$$z(D)=\left[\frac{\sum_{i=1}^N\phi(\mathbf R(x_i))-N\,\overline\phi_{\mathrm{center}}}{\sqrt{\max(N,1)}},\frac{N-B}{\sqrt B},y\right].$$

This is a centered, scaled mean pool with $N$ retained. Centering uses a fixed subset of the background bank and is recomputed through the trainable encoder; it changes conditioning, not which events are included. The empty set has a zero event summary.

The global head receives $z$ and outputs $(\widehat\nu,\widehat\alpha)$. The conditional head also receives a tested **$\nu$** and outputs $\widehat\alpha_\nu$. The fitted nuisance is an output, not an externally supplied input. With several POIs/nuisances, these inputs and outputs become vectors.

We train the encoder and both heads together using

$$\mathcal L=\mathbb E_D\left[\mathrm{NLL}(\widehat\nu(D),\widehat\alpha(D);D)+\mathbb E_\nu\mathrm{NLL}(\nu,\widehat\alpha_\nu(D);D)\right].$$

The loss sums **all events in each training experiment**. It includes the count/rate term and Gaussian constraint exactly once. No scalar test-statistic labels, fit labels, or quadratic approximation enter this objective. A finite learned pool can still discard useful information, so direct-fit validation is essential.

### Training ensembles and cost

We use independent experiments from both the physical simulator and the bad reference model so the optimizer encounters both domains. This is not required by the likelihood formula, but helps its approximation cover the two distributions used in notebook 05. Training generating nuisances span $[-1,1]$; the calibration study remains at physical $\alpha_{\mathrm{gen}}=0$.

The reference training toys resample the integration bank, an explicit finite-bank approximation used to avoid repeatedly generating and evaluating large flow proposals. Physical simulator toys are newly generated. Final validation below uses **new physical unbinned events**, not this bank.

The default is 2,000 training experiments per source, with $N$ around 10,000 each. The six float32 ratios require about 1 GB for the full training cache. Start with 128–512 per source to check a setup if needed, then increase the setting for the larger study. Training-toy caches and profile checkpoints are keyed by sample count, so increasing it creates a fresh training stage. More training toys can reduce amortization error, but the held-out likelihood gaps decide whether they are sufficient.

In [ ]:
centering_anchors = background_bank[:2048]
N_TRAIN_PER_SOURCE = 2000
N_MONITOR_PER_SOURCE = 128
PROFILE_EPOCHS = 60
RETRAIN_PROFILES = False

toys_path = OUT / f"training_experiments_{N_TRAIN_PER_SOURCE}_{N_MONITOR_PER_SOURCE}.pt"
if not toys_path.exists():
    training_toys = make_training_experiments(
        hybrid, config, epsilon, N_TRAIN_PER_SOURCE,
        np.random.default_rng(config["seed"] + 4301))
    monitor_toys = make_training_experiments(
        hybrid, config, epsilon, N_MONITOR_PER_SOURCE,
        np.random.default_rng(config["seed"] + 4302))
    torch.save({"training": training_toys, "monitor": monitor_toys}, toys_path)
else:
    cached_toys = torch.load(toys_path, map_location="cpu", weights_only=False)
    training_toys, monitor_toys = cached_toys["training"], cached_toys["monitor"]
print("Training experiments:", len(training_toys), "| monitoring experiments:", len(monitor_toys))
print("Cached events:", sum(len(t["anchors"]) for t in training_toys))

profile_path = OUT / f"profiles_{N_TRAIN_PER_SOURCE}toys_{PROFILE_EPOCHS}epochs.pt"
profile_history_path = profile_path.with_suffix(".json")
if RETRAIN_PROFILES or not profile_path.exists():
    profile_network, profile_history = train_profiles(
        training_toys, monitor_toys, centering_anchors, config,
        epochs=PROFILE_EPOCHS, batch_size=8, device=device,
        seed=config["seed"] + 4303)
    torch.save(profile_network.cpu().state_dict(), profile_path)
    profile_history_path.write_text(json.dumps(profile_history, indent=2))
else:
    profile_network = ProfileNetwork(centering_anchors, config)
    profile_network.load_state_dict(torch.load(profile_path, map_location="cpu", weights_only=True))
    profile_history = json.loads(profile_history_path.read_text())

inference = Inference(response_network, profile_network, config, epsilon, device)
save_inference(RUN, inference, metadata={**spec,
    "training_experiments": len(training_toys),
    "monitor_experiments": len(monitor_toys),
    "response_population_per_process": len(signal_bank),
    "auxiliary_law": "Uniform[-2,2]", "physical_calibration_alpha": 0.0,
    "reference_training": "finite integration bank",
    "statistic": "signed full-likelihood gap at amortized optima"})

hist = pd.DataFrame(profile_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist.epoch, hist.train)
axes[0].set(title="Training queries", xlabel="Epoch", ylabel="Centered global + conditional NLL")
axes[1].plot(hist.epoch, hist.validation)
axes[1].set(title="Fixed held-out queries (best checkpoint retained)", xlabel="Epoch", ylabel="Centered global + conditional NLL")
plt.tight_layout()
plt.show()

## 3. Obtain a statistic without an iterative profile fit

For any tested $\nu$, evaluate the original likelihood at the two predicted optima:

$$t_{\mathrm{raw}}(\nu;D)=2\left[\mathrm{NLL}(\nu,\widehat\alpha_\nu(D);D)-\mathrm{NLL}(\widehat\nu(D),\widehat\alpha(D);D)\right].$$

The exact constrained fits would make this nonnegative. Approximate optimizers can produce negative values. We retain and report these **signed optimization gaps** here; no clipping conceals an optimization failure.

At evaluation, a small fixed candidate list includes $\alpha=-1,0,+1$ and global POI boundaries, alongside network predictions. This allows exact endpoints and the interpolation kink to win despite bounded smooth NN outputs. It is not an iterative minimization, and it does not guarantee the global optimum.

The encoder and the final likelihood evaluation still scale with the number of events. A scan reuses one experiment embedding, but each query evaluates the full likelihood. What we amortize is the search for fitted parameters. We do not claim constant-cost likelihood evaluation or an exact sufficient learned embedding.

## 4. Validate on actual unbinned experiments

The checks below compare to direct branch-aware fits of the same likelihood. We inspect global and conditional NLL excess **separately**, because errors in the two optima can cancel in their difference. We also inspect fitted parameters, statistic tails, negative gaps, and full nonlinear scan shapes. Evaluating the statistic at its own global estimate should give approximately zero; a positive value exposes inconsistency between the global and conditional nuisance heads.

The timing comparison excludes simulation and ratio evaluation for both methods, and includes embedding and final likelihood evaluation for the NN. This toy has only two fitted parameters, so a numerical fit may already be fast; any speed advantage must be measured.

In [ ]:
VALIDATION_MU = [0., 0.25, 1., 2.]
N_VALIDATION_PER_MU = 32
validation_rows, scan_examples = validate_toys(
    inference, hybrid, config, epsilon, VALIDATION_MU, N_VALIDATION_PER_MU,
    np.random.default_rng(config["seed"] + 4401), alpha_gen=0.)
validation = pd.DataFrame(validation_rows)
validation["statistic_error"] = validation.statistic_nn - validation.statistic_exact
validation.to_csv(OUT / "unbinned_validation.csv", index=False)
torch.save(scan_examples, OUT / "scan_examples.pt")

summary = pd.DataFrame([{
    "mu": mu, "toys": len(group),
    "median_abs_T_error": group.statistic_error.abs().median(),
    "p95_abs_T_error": group.statistic_error.abs().quantile(.95),
    "median_global_NLL_gap": group.global_gap.median(),
    "p95_global_NLL_gap": group.global_gap.quantile(.95),
    "median_conditional_NLL_gap": group.conditional_gap.median(),
    "p95_conditional_NLL_gap": group.conditional_gap.quantile(.95),
    "negative_T_fraction": (group.statistic_nn < 0).mean(),
    "minimum_raw_T": group.statistic_nn.min(),
    "p95_T_at_own_minimum": group.statistic_at_own_minimum.quantile(.95),
    "direct_fit_success_fraction": group.fit_success.mean(),
} for mu, group in validation.groupby("mu")])
display(summary)
print("Median NN evaluation seconds:", validation.nn_seconds.median())
print("Median direct-fit seconds:", validation.exact_seconds.median())
print("Median time ratio direct / NN:", (validation.exact_seconds / validation.nn_seconds).median())
print("Negative raw gaps:", int((validation.statistic_nn < 0).sum()), "/", len(validation))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].scatter(validation.statistic_exact, validation.statistic_nn, c=validation.mu, s=15)
limit = max(validation.statistic_exact.max(), validation.statistic_nn.max())
axes[0, 0].plot([0, limit], [0, limit], "k--")
axes[0, 0].set(xlabel="Direct unbinned t", ylabel="NN signed t")
axes[0, 1].scatter(validation.statistic_exact, validation.statistic_error, s=15)
axes[0, 1].axhline(0, color="k", ls="--")
axes[0, 1].set(xlabel="Direct unbinned t (inspect tails)", ylabel="NN t − direct t")
axes[0, 2].hist(validation.global_gap, bins=25, alpha=.6, label="Global")
axes[0, 2].hist(validation.conditional_gap, bins=25, alpha=.6, label="Conditional")
axes[0, 2].set(xlabel="NN NLL − direct minimum", ylabel="Toys")
axes[0, 2].legend()
for ax, name, bounds in [(axes[1, 0], "nu", config["nu_bounds"]),
                         (axes[1, 1], "alpha", [-1, 1]),
                         (axes[1, 2], "conditional_alpha", [-1, 1])]:
    ax.scatter(validation[name + "_exact"], validation[name + "_nn"], s=15)
    ax.plot(bounds, bounds, "k--")
    ax.set(xlabel="Direct " + name, ylabel="NN " + name)
plt.tight_layout()
plt.show()

### Boundary and tail summaries

Boundary fits and large statistics deserve separate attention. The numerical boundary tolerance below only labels results for this diagnostic; it does not alter any fit or statistic. Small samples in the top tail give noisy summaries, so increase the independent validation sample before making a quantitative tail-accuracy claim.

In [ ]:
boundary = (np.isclose(validation.nu_exact, config["nu_bounds"][0], atol=1e-5)
            | np.isclose(validation.nu_exact, config["nu_bounds"][1], atol=1e-5)
            | np.isclose(np.abs(validation.alpha_exact), 1., atol=1e-5)
            | np.isclose(np.abs(validation.conditional_alpha_exact), 1., atol=1e-5))
tail_threshold = validation.statistic_exact.quantile(.90)
subsets = {"all": np.ones(len(validation), dtype=bool),
           "boundary fits": boundary,
           "top 10% direct t": validation.statistic_exact >= tail_threshold}
display(pd.DataFrame([{"subset": name, "toys": int(mask.sum()),
    "mean_abs_T_error": validation.loc[mask, "statistic_error"].abs().mean(),
    "max_abs_T_error": validation.loc[mask, "statistic_error"].abs().max(),
    "mean_global_NLL_gap": validation.loc[mask, "global_gap"].mean(),
    "mean_conditional_NLL_gap": validation.loc[mask, "conditional_gap"].mean()}
    for name, mask in subsets.items()]))

In [ ]:
nu_scan = np.linspace(*config["nu_bounds"], 61)
fig, axes = plt.subplots(1, len(scan_examples), figsize=(5 * len(scan_examples), 4))
scan_records = []
for ax, toy in zip(np.atleast_1d(axes), scan_examples):
    estimate = inference.evaluate(toy["anchors"], toy["auxiliary"], nu_scan)
    best = fit(toy["anchors"], toy["auxiliary"], config)
    exact = np.asarray([2 * (fit(toy["anchors"], toy["auxiliary"], config, fixed_nu=nu)["nll"] - best["nll"])
                        for nu in nu_scan])
    ax.plot(nu_scan, exact, label="Direct full likelihood")
    ax.plot(nu_scan, estimate["statistic"], "--", label="Amortized optima")
    ax.axhline(0, color="gray", lw=.7)
    ax.set(xlabel=r"$\nu$", ylabel=r"$t(\nu)$", title=f'Physical mu={toy["mu"]:g}, y={toy["auxiliary"]:.2f}')
    scan_records.append({"mu": toy["mu"], "nu": nu_scan.tolist(), "direct_t": exact.tolist(),
                         "nn_raw_t": estimate["statistic"].tolist()})
axes[0].legend()
plt.tight_layout()
plt.show()
(OUT / "scan_validation.json").write_text(json.dumps(scan_records, indent=2))

## What to inspect before the distribution flows

- **Response:** same-bank gaps diagnose optimization; disagreement between independent population fits diagnoses finite-bank precision. Inspect both $g$ and the fitted nuisance $a$.
- **Profiling:** compare conditional/global NLL excess separately, including nuisance boundaries and the interpolation kink. If training improves but held-out gaps do not, add training experiments; if both remain poor, change optimization or encoder/head capacity.
- **Statistic:** inspect signed gaps and scan shapes, especially tails. A small average error does not establish inference accuracy at a particular tail probability.
- **Cost:** record the measured speed rather than assuming the NN helps in this simple two-parameter example.

Notebook 05 calibrates the distribution of the learned statistic on separate toys. It explicitly defines $T_{\mathrm{NN}}=\max(0,t_{\mathrm{raw}})$ and models its point mass at zero; optimization-induced zeros are tracked separately from genuine constrained-fit effects. Calibrating that distribution can yield valid tests for the chosen toy ensemble, but it does not turn an inaccurate optimizer into the exact profile likelihood. The raw-gap and direct-fit checks in this notebook remain necessary.